# 01 — Load and explore

Tier 1 POTS/dysautonomia findings: load the data, check its shape, measure every field's fill
rate yourself, and take a first look at conditions, entities and evidence tiers.

Runs in Colab with no setup. Dependencies: pandas, numpy, matplotlib, pyarrow.

> ## Scope — read this first
>
> **This is Tier 1: POTS / dysautonomia literature ONLY** — chunks 0–75 of a 286-chunk corpus,
> **~3,121 of ~14,263 corpus papers**.
>
> ME/CFS, fibromyalgia, EDS/hypermobility, MCAS, IBS, migraine and autism appear here **only as
> incidental mentions inside POTS/dysautonomia papers**. Those conditions' own literatures are in
> Tiers 2–4 and are **not yet extracted**. This is **not** a cross-condition resource.
>
> **Not medical advice.** Machine-extracted research information, not clinical guidance. No
> diagnostic or treatment recommendations. Personal health questions belong with a qualified
> clinician.
>
> **Attribution.** Every finding links to its source paper with authors and journal. These are
> automated summaries of other researchers' work — the credit is theirs, and the original paper is
> the authority. `provenance_quote_trunc` is short verbatim publisher-copyright text included for
> verification only; always keep it next to its citation and never bulk-export it.

In [1]:
# ── Data location ──────────────────────────────────────────────────────────────
# Repoint this constant at your own fork if you want. Raw GitHub is used in Colab;
# a local ./data or ../data directory (i.e. running inside a clone) wins if present.
RAW_BASE = "https://raw.githubusercontent.com/USER/REPO/main/data"

import os, io, json, urllib.request
import pandas as pd, numpy as np
import matplotlib
import matplotlib.pyplot as plt

_LOCAL = next((p for p in ("data", "../data", "../../data") if os.path.isdir(p)), None)
_CACHE = ".cache_tier1"; os.makedirs(_CACHE, exist_ok=True)

def data_path(name):
    """Return a readable path for a data file: local clone if present, else download once."""
    if _LOCAL and os.path.exists(os.path.join(_LOCAL, name)):
        return os.path.join(_LOCAL, name)
    dst = os.path.join(_CACHE, name)
    if not os.path.exists(dst):
        url = f"{RAW_BASE}/{name}"
        print(f"downloading {name} ...")
        urllib.request.urlretrieve(url, dst)
    return dst

def load(name):
    p = data_path(name)
    if p.endswith(".parquet"): return pd.read_parquet(p)
    if p.endswith(".json"):    return json.load(open(p))
    return pd.read_csv(p, low_memory=False)

print("data source:", _LOCAL if _LOCAL else RAW_BASE)

data source: ../data


## 1. Load

In [2]:
F = load("tier1_findings.parquet")
print("shape:", F.shape)
print(f"{F.shape[1]} columns")
F.head(3)[["canonical_pmid","year","condition","canonical_entity","direction","evidence_tier","finding_text"]]

shape: (36459, 89)
89 columns


## 2. Counting rules — get these right before anything else

The database has two paper-id columns and a row-inflation flag. Using the wrong one silently
changes your denominators.

In [3]:
# ── The counting rule, as a function. Use it everywhere. ──────────────────────
def n_papers(df):
    """Correct paper count: distinct canonical_pmid (3,121 corpus-wide, NOT 3,129)."""
    return df.canonical_pmid.nunique()

def n_papers_direct(df):
    """Paper count excluding split_derived rows (compound-entity splits inflate entity counts)."""
    return df.loc[~df.split_derived, "canonical_pmid"].nunique()

print("findings rows      :", f"{len(F):,}")
print("distinct papers    :", f"{n_papers(F):,}   (canonical_pmid — the correct number)")
print("distinct pmid       :", f"{F.pmid.nunique():,}   (do NOT use for paper counts)")
print("papers_direct      :", f"{n_papers_direct(F):,}")
print("split_derived rows :", f"{int(F.split_derived.sum()):,}")
print("distinct finding_text:", f"{F.finding_text.nunique():,}")

findings rows      : 36,459
distinct papers    : 3,121   (canonical_pmid — the correct number)
distinct pmid       : 3,129   (do NOT use for paper counts)
papers_direct      : 3,121
split_derived rows : 696
distinct finding_text: 36,032


### Why 3,129 vs 3,121

Nine normalized titles span more than one PMID (18 PMIDs, 179 findings). All nine groups were
**examined**; 8 were merged (53 findings reattributed via `canonical_pmid`, none deleted, fully
reversible) and **1 was deliberately retained** — two genuinely distinct papers sharing the
generic title "Post-COVID-19 Postural Orthostatic Tachycardia Syndrome" with fully disjoint
author lists. Title-only matching would have merged them and destroyed a real independent
replication.

In [4]:
DD = load("pmid_dedup_map.parquet")
print("duplicate-title groups examined:", DD.title_norm.nunique())
print("groups merged                 :", int(DD.groupby('title_norm').merge_applied.max().sum()))
print("pmids folded into a canonical  :", int((DD.pmid != DD.canonical_pmid).sum()))
print("findings reattributed          :", int(DD.loc[DD.pmid != DD.canonical_pmid, 'n_findings'].sum()))
print()
print("the group that was NOT merged (author_jaccard = 0.0 -> distinct papers):")
DD[~DD.merge_applied.groupby(DD.title_norm).transform('max')][
    ["pmid","title","journal","year","author_jaccard","merged_reason"]]

duplicate-title groups examined: 9
groups merged                 : 8
pmids folded into a canonical  : 8
findings reattributed          : 53

the group that was NOT merged (author_jaccard = 0.0 -> distinct papers):


## 3. Fill rates — measured, not quoted

This recomputes every field's fill rate from the parquet. Empty strings count as missing.

In [5]:
def fill_rates(df):
    out = []
    for c in df.columns:
        s = df[c]
        nn = (s.notna() & (s.astype(str).str.strip() != "")) if s.dtype == object else s.notna()
        out.append((c, str(s.dtype), round(100 * nn.sum() / len(df), 2), int(s[nn].nunique())))
    return pd.DataFrame(out, columns=["field","dtype","fill_pct","n_unique"]).sort_values("fill_pct")

FR = fill_rates(F)
print("fields at 100%:", int((FR.fill_pct >= 99.995).sum()), "| under 15%:", int((FR.fill_pct < 15).sum()))
FR.head(15)

fields at 100%: 34 | under 15%: 24


### The two fill rates that change how you read everything else

`canonical_entity` covers only **29.8%** of rows (10,858 of 36,459), while the original `entity`
string covers **53.0%** (19,307). Two different columns, two different numbers. 47.0% of rows name
no entity at all — those are symptom-burden, phenotype and
study-level findings that name no single canonical entity. They are not extraction failures.

But there is a **second, separate gap**: a further **8,449 rows do carry an entity string that
never received a canonical mapping**, because canonicalization was head-only by deliberate scope
decision (1,803 typed vocabulary entries against 9,582 distinct strings; 7,605 left unmapped).

So **entity-keyed analysis below covers ~30% of findings, spanning 2,185 of 3,121 papers — not
half the table.** The 8,449 unmapped-string rows are real findings: search them by keyword on
`entity` / `entity_norm_original`. Absence from an entity rollup is not absence from the corpus.

In [6]:
for c in ["entity","canonical_entity","canonical_ontology_id","map_confidence"]:
    print(f"{c:24s} {FR.set_index('field').loc[c,'fill_pct']:6.2f}% filled")
print()
# The gap between entity (53.0%) and canonical_entity (29.8%) has TWO separate causes.
_e = F.entity.notna() & (F.entity.astype(str).str.strip() != "")
_c = F.canonical_entity.notna() & (F.canonical_entity.astype(str).str.strip() != "")
print(f"entity string present        : {int(_e.sum()):>7,}  ({100*_e.mean():4.1f}%)")
print(f"canonical_entity present     : {int(_c.sum()):>7,}  ({100*_c.mean():4.1f}%)   <- the number for entity views")
print(f"entity string, NO canonical  : {int((_e & ~_c).sum()):>7,}  ({100*(_e & ~_c).mean():4.1f}%)   <- real findings, not rolled up")
print(f"no entity at all             : {int((~_e).sum()):>7,}  ({100*(~_e).mean():4.1f}%)   <- by design")
print()
print(f"papers with >=1 canonical entity: {F.loc[_c,'canonical_pmid'].nunique():,} of {F.canonical_pmid.nunique():,}")
print()
print("distinct normalized entity strings :", F.loc[_e,'entity_norm_original'].nunique())
print("  ...that carry a canonical mapping:", F.loc[_e & _c,'entity_norm_original'].nunique())
print("  ...left unmapped (head-only)     :", F.loc[_e & ~_c,'entity_norm_original'].nunique())
_tail = F.loc[_e & ~_c, 'entity_norm_original'].value_counts()
print(f"  unmapped tail: {int((_tail==1).sum()):,} singletons, max {int(_tail.max())} rows for any one string")
print()
print("rows with no entity at all, by finding_type:")
print(F.loc[~_e, "finding_type"].value_counts().head(8))

entity                    52.96% filled
canonical_entity          29.78% filled
canonical_ontology_id     11.85% filled
map_confidence            29.78% filled

entity string present        :  19,307  (53.0%)
canonical_entity present     :  10,858  (29.8%)   <- the number for entity views
entity string, NO canonical  :   8,449  (23.2%)   <- real findings, not rolled up
no entity at all             :  17,152  (47.0%)   <- by design

papers with >=1 canonical entity: 2,185 of 3,121

distinct normalized entity strings : 9582
  ...that carry a canonical mapping: 1977
  ...left unmapped (head-only)     : 7605
  unmapped tail: 6,761 singletons, max 2 rows for any one string

rows with no entity at all, by finding_type:
finding_type
mechanism/biomarker       4794
treatment/intervention    3388
symptom-burden            3224
comorbidity               2986
diagnostic/screening      2760
Name: count, dtype: int64


### Thin enrichment fields — sparse because the papers don't say it

These are **not** a pipeline bug and re-extraction will not improve them. Papers do not state
GO/Reactome IDs, most clinical studies deposit no dataset, HPO terms are not clinical-paper
vocabulary, and papers rarely name a specific finding they contradict. Raising these needs
external database joins keyed on the canonical entities. **Never use them as analysis keys.**

In [7]:
thin = ["pathway_id","dataset_accession","phenotype_hpo","contradicts_finding_ref","entity_ontology_id"]
print(FR.set_index("field").loc[thin, ["fill_pct","n_unique"]])
print()
print("the mechanistic content is in prose instead:")
print("  pathway_or_mechanism", FR.set_index('field').loc['pathway_or_mechanism','fill_pct'], "% filled")
print()
print("source_fidelity is single-valued in Tier 1 - it carries ZERO information:")
print(F.source_fidelity.value_counts())

                         fill_pct  n_unique
field                                      
pathway_id                   0.06        15
dataset_accession            0.72        52
phenotype_hpo                2.78       196
contradicts_finding_ref      4.17      1493
entity_ontology_id          12.67      1560

the mechanistic content is in prose instead:
  pathway_or_mechanism 63.9 % filled

source_fidelity is single-valued in Tier 1 - it carries ZERO information:
source_fidelity
high    36459
Name: count, dtype: int64


## 4. First look — conditions, tiers, entities

**Scope reminder:** non-POTS condition counts below are POTS/dysautonomia papers that *mention*
that condition. They are not that condition's literature.

In [8]:
cond = pd.DataFrame({
    "findings": F.condition.value_counts(),
    "papers":   F.groupby("condition").canonical_pmid.nunique(),
}).sort_values("findings", ascending=False)
print(cond)
print("\n^ non-POTS rows = incidental mentions inside POTS/dysautonomia papers, NOT their own literature")

                         findings  papers
condition                                
dysautonomia                18293    2332
general                      5854     853
POTS                         5078     749
long_COVID                   2608     324
multiple                     1129     345
ME_CFS                        845     136
fibromyalgia                  756     103
small_fiber_neuropathy        642     202
EDS_hypermobility             625     146
migraine                      255      61
MCAS                          236      90
IBS_gut                        90      64
autism                         44      18
orthostatic_hypotension         4       1

^ non-POTS rows = incidental mentions inside POTS/dysautonomia papers, NOT their own literature


In [9]:
print("evidence_tier:"); print(F.evidence_tier.value_counts())
print("\nRCT + meta-analysis share:",
      f"{100*F.evidence_tier.isin(['RCT','meta-analysis']).mean():.1f}%  ({int(F.evidence_tier.isin(['RCT','meta-analysis']).sum()):,} rows)")
print("\nfinding_type:"); print(F.finding_type.value_counts())
print("\ncertainty:"); print(F.certainty.value_counts())
print("\ncausal_or_correlational:"); print(F.causal_or_correlational.value_counts())

evidence_tier:
evidence_tier
cohort                   8239
cross-sectional          7009
review/expert-opinion    6507
case-report              3777
case-series              3239
case-control             3184
mechanistic/in-vitro     2787
RCT                       877
meta-analysis             840
Name: count, dtype: int64

RCT + meta-analysis share: 4.7%  (1,717 rows)

finding_type:
finding_type
mechanism/biomarker       16348
treatment/intervention     7210
diagnostic/screening       4830
comorbidity                4187
symptom-burden             3884
Name: count, dtype: int64

certainty:
certainty
confirmed    23310
hedged        9347
mixed         3802
Name: count, dtype: int64

causal_or_correlational:
causal_or_correlational
correlational-only    23193
not-stated             8893
causal-claim           4373
Name: count, dtype: int64


In [10]:
ent = F[F.canonical_entity.notna()]
can_n = len(ent)   # denominator for every entity-keyed chart below
top = (ent.groupby(["canonical_entity","entity_kind"])
          .agg(papers=("canonical_pmid","nunique"), findings=("finding_text","size"))
          .sort_values("papers", ascending=False).head(20))
print("Top 20 canonical entities by distinct papers")
print("(from the 29.8% of rows carrying a canonical_entity — not the whole table,")
print(" and not the 53.0% that carry an entity string)")
print(top)

Top 20 canonical entities by distinct papers
(from the 29.8% of rows carrying a canonical_entity — not the whole table,
 and not the 53.0% that carry an entity string)
                                                       papers  findings
canonical_entity           entity_kind                                 
norepinephrine             molecule-or-analyte            194       270
heart rate                 physiological-measure          178       271
heart rate variability     physiological-measure          170       291
orthostatic hypotension    clinical-sign-or-condition     115       249
midodrine                  drug-or-drug-class              95       153
sdnn                       physiological-measure           94       162
rmssd                      physiological-measure           84       121
lf/hf ratio                physiological-measure           81       117
snca                       gene-or-protein                 77       182
il-6                       molecule-or-a

In [11]:
fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.4))
vc = F.condition.value_counts().head(10)[::-1]
axes[0].barh(range(len(vc)), vc.values, color="#4C72B0")
axes[0].set_yticks(range(len(vc))); axes[0].set_yticklabels(vc.index, fontsize=8)
axes[0].set_xlabel("findings"); axes[0].set_title("Findings by condition label", fontsize=9, loc="left")
vt = F.evidence_tier.value_counts()[::-1]
axes[1].barh(range(len(vt)), vt.values, color="#55A868")
axes[1].set_yticks(range(len(vt))); axes[1].set_yticklabels(vt.index, fontsize=8)
axes[1].set_xlabel("findings"); axes[1].set_title("Study design tier", fontsize=9, loc="left")
tp = top.papers.head(12)[::-1]
axes[2].barh(range(len(tp)), tp.values, color="#C44E52")
axes[2].set_yticks(range(len(tp)))
axes[2].set_yticklabels([f"{e} ({k})" for e, k in tp.index], fontsize=7)
axes[2].set_xlabel("distinct papers"); axes[2].set_title(f"Top entities ({100*can_n/len(F):.1f}% of findings\nhave a canonical_entity)", fontsize=9, loc="left")
fig.tight_layout(); fig.savefig("01_first_look.png", dpi=110)
print("values plotted, left to right:")
print(dict(vc[::-1])); print(dict(vt[::-1])); print(dict(tp[::-1]))

values plotted, left to right:
{'dysautonomia': np.int64(18293), 'general': np.int64(5854), 'POTS': np.int64(5078), 'long_COVID': np.int64(2608), 'multiple': np.int64(1129), 'ME_CFS': np.int64(845), 'fibromyalgia': np.int64(756), 'small_fiber_neuropathy': np.int64(642), 'EDS_hypermobility': np.int64(625), 'migraine': np.int64(255)}
{'cohort': np.int64(8239), 'cross-sectional': np.int64(7009), 'review/expert-opinion': np.int64(6507), 'case-report': np.int64(3777), 'case-series': np.int64(3239), 'case-control': np.int64(3184), 'mechanistic/in-vitro': np.int64(2787), 'RCT': np.int64(877), 'meta-analysis': np.int64(840)}
{('norepinephrine', 'molecule-or-analyte'): np.int64(194), ('heart rate', 'physiological-measure'): np.int64(178), ('heart rate variability', 'physiological-measure'): np.int64(170), ('orthostatic hypotension', 'clinical-sign-or-condition'): np.int64(115), ('midodrine', 'drug-or-drug-class'): np.int64(95), ('sdnn', 'physiological-measure'): np.int64(94), ('rmssd', 'physiol

## 5. Verify a finding against its source

This is the point of the grounding columns. Any row can be checked in under a minute.

In [12]:
row = F[(F.evidence_tier == "RCT") & F.canonical_entity.notna()].iloc[0]
for k in ["finding_text","canonical_entity","direction","evidence_tier","certainty",
          "citation_short","journal","year","provenance_section","pubmed_url","doi_url"]:
    print(f"{k:20s}: {row[k]}")
print(f"{'quote (<=200 chars)':20s}: {row['provenance_quote_trunc']}")
print("\n^ short verbatim excerpt, publisher copyright, shown here beside its full citation and link.")

finding_text        : Supine and upright diastolic blood pressure increased after bedrest in the control group but not in the LBNP, cycling, or seated intervention groups, indicating an interaction whereby all countermeasures attenuated this bedrest effect.
canonical_entity    : diastolic blood pressure
direction           : elevated
evidence_tier       : RCT
certainty           : confirmed
citation_short      : Hönemann JN et al. (2024) Journal of the American Heart Association
journal             : Journal of the American Heart Association
year                : 2024.0
provenance_section  : results
pubmed_url          : https://pubmed.ncbi.nlm.nih.gov/39450723/
doi_url             : https://doi.org/10.1161/jaha.124.034800
quote (<=200 chars) : In contrast, supine and upright diastolic blood pressure increased in the control group but not in the intervention groups. The significant interaction indicates that all interventions attenuated bed…

^ short verbatim excerpt, publisher copyrig

## Next

- `02_reproduce_figures.ipynb` — every published figure, with its numbers printed beside it
- `03_associations_and_contradictions.ipynb` — the four discovery tables
- `04_query_recipes.ipynb` — 12 practical queries with caveats inline